# Lab: Hybrid RAG (Dense + Sparse) with Semantic Chunking

## Setup

### Install Dependencies

In [ ]:
!pip install llama-index-core llama-index-llms-openrouter llama-index-embeddings-huggingface llama-index-readers-file python-dotenv pymupdf rank_bm25

### Import Libraries

In [1]:
import os
from dotenv import load_dotenv

# Core LlamaIndex components
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.readers.file import PDFReader
from llama_index.core.query_engine import RetrieverQueryEngine

# Hybrid Search Specifics
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

# LLM and Embedding integrations
from llama_index.llms.openrouter import OpenRouter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

### Load API Keys & Configure LlamaIndex Settings

In [4]:
import warnings
import logging
# 2. Silence general Python warnings
warnings.filterwarnings("ignore")

# 3. Silence HuggingFace download/info logs
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)


In [5]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Key loaded.")

# Set up the LLM (using OpenRouter)
Settings.llm = OpenRouter(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    max_tokens=512
)

# Set up the dense embedding model
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

print("LlamaIndex configured!")

Key loaded.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LlamaIndex configured!


---
## Step 1 — Load Documents & Create Semantic Nodes

In [7]:
# Load PDF document
reader = PDFReader()

# Remove the fake citation tag here!
documents = reader.load_data(file="data/sample_text_document.pdf")

print(f"Loaded {len(documents)} document(s)")

# Semantic chunking: splits based on meaning, not fixed size
splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percentile_threshold=95, 
    embed_model=Settings.embed_model
)

# Parse documents into nodes explicitly for hybrid search
nodes = splitter.get_nodes_from_documents(documents)

print(f"Extracted {len(nodes)} semantic nodes from the documents.")

Loaded 2 document(s)
Extracted 4 semantic nodes from the documents.


---
## Step 2 — Create the Hybrid Retrievers (Dense + Sparse)

Here we define our two retrieval mechanisms:
1. **Vector Retriever (Dense):** Finds context based on semantic meaning.
2. **BM25 Retriever (Sparse):** Finds context based on exact keyword matches.

We then fuse them together using `QueryFusionRetriever`.

In [22]:
# 1. THE DENSE COMPONENT
vector_index = VectorStoreIndex(nodes)
vector_retriever = vector_index.as_retriever(similarity_top_k=3)

# 2. THE SPARSE COMPONENT
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes, 
    similarity_top_k=3
)

# 3. THE HYBRID ENGINE (Fusing them together)
hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1, 
    mode="reciprocal_rerank", 
)

print("Hybrid retrievers (Dense + Sparse) successfully fused!")

2026-07-21 23:24:14,539 - DEBUG - Building index from IDs objects


Hybrid retrievers (Dense + Sparse) successfully fused!


---
## Step 3 — Query the Index

In [23]:
# Create a query engine from the fused retriever
query_engine = RetrieverQueryEngine.from_args(hybrid_retriever)

# Ask a question
QUERY = "Why do restoration teams reintroduce tidal flow gradually instead of all at once?"
response = query_engine.query(QUERY)

print(f"Query: {QUERY}")
print(f"\nAnswer: {response}")

Query: Why do restoration teams reintroduce tidal flow gradually instead of all at once?

Answer: Restoration teams reintroduce tidal flow gradually because a sudden, full breach can erode unconsolidated fill material before vegetation has a chance to stabilize the soil. A phased approach—opening a small channel first, then widening it in stages—allows time for pioneer plant species to colonize newly wetted areas and for teams to monitor erosion and sediment deposition before proceeding further.


---
## Step 4 — Inspect Post-Fusion Sources

In [24]:
# Show the source nodes (retrieved chunks)
print("Source nodes used (Post-Fusion):")
for i, node in enumerate(response.source_nodes):
    print(f"\n--- Source {i + 1} (score: {node.score:.4f}) ---")
    print(node.text[:200] + "...")

Source nodes used (Post-Fusion):

--- Source 1 (score: 0.0333) ---
A Short Guide to Coastal Wetland Restoration
Synthetic Sample Document — Original Content, No Copyright Restrictions
1. Introduction
Coastal wetlands sit at the boundary between land and sea, absorbin...

--- Source 2 (score: 0.0328) ---
4. Planting and Vegetation Recovery
Many restoration sites are left to revegetate naturally once tidal flow and elevation are corrected, since
wind- and water-borne seeds from nearby healthy marsh oft...

--- Source 3 (score: 0.0161) ---
Phased approaches might open a small channel first, monitor erosion and sediment
deposition for a season, and then widen the opening in stages. This measured pace gives pioneer plant
species time to c...
